# GFTM Galerkin field solve: does it make the low-NBASIS max-$\gamma$ width selection reliable on the GS2-KBM cases of R3, M1 and R4?

Fusion_PhD-qp5.24. GFTM's EM field response uses the pointwise quasi-inverses $\mathrm{Mat}(J/p_0)$, $\mathrm{Mat}(J/b_0)$
(`FIELD_GALERKIN F`, production behaviour). The patch `tools/gftm_galerkin_field/gftm_galerkin_field.patch` replaces them
with the Galerkin $\mathrm{Mat}(p_0)^{-1}\mathrm{Mat}(J)$, $\mathrm{Mat}(b_0)^{-1}\mathrm{Mat}(J)$ (`FIELD_GALERKIN T`). On the
laptop (R3, one width, `Fusion_PhD-qp5.23`) this made $D\cdot B$ Hermitian and cut the NBASIS-8 self-convergence error at
ky 0.1, but not at ky 0.3/0.6, where the dominant mode changed identity with NBASIS.

**Data.** Both arms ran on the Galerkin build (md5 `979f191a…`; `=F` is bit-identical to production).
Leaves `GFTM/Runs/LATIN_HYPERCUBE/<case>/kbm_free/gal{F,T}_w<W>_nb<N>_nx<X>/`, registry family `gftm_galerkin_field`:
NBASIS 2/4/8/16 at NXGRID 32 and NBASIS 64 at NXGRID 65, each at the 24 widths 0.2-4.0, FILTER 0.5, IBRANCH $-1$, NMODES 4,
USE_BPAR from the deck. The population is the GS2-KBM samples of each `kbm_free` GS2 cube (user classifier,
docs/mode_filtering.md; $T$ from the cube's saved `tearing_parameter`; converged = $\gamma > 0$ and
`growth_rate_tolerance` < 0.05), selected by Fusion_PhD `results/gftm_galerkin_field/population.py`: GFTM was run on those
samples only, so the population is the samples in each GFTM leaf. GS2 reference: the cube's growth rate and frequency
(pyro's default `time_mode="average"`: the mean over the last 20% of the run) and the time-averaged $|\phi|^2$.

**Dominant mode**: IBRANCH $-1$ returns the modes ranked by growth rate, so mode 0 is the dominant one (checked on every
leaf below). A run with no unstable mode counts as $\gamma = 0$; its frequency, extent and parity drop out.

## Imports and settings

In [ ]:
from pathlib import Path
import os
import json
import re
import warnings
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import pyrokinetics
from pyrokinetics import PyroHypercube
from pyrokinetics.diagnostics.extent import Extent

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
print("pyrokinetics", pyrokinetics.__version__)

analysis_name = "gftm_galerkin_field"
gs2_root = Path(os.environ["GK_DATA_ROOT"]).expanduser() / "GS2" / "Runs" / "LATIN_HYPERCUBE"
# GyroRun wrote this scan to the scratch tree; it has no copy under GK_DATA_ROOT
gftm_root = Path(os.environ["GYRO_DATA_OUTPUT"]) / "GFTM" / "Runs" / "LATIN_HYPERCUBE"
cases = ["R3", "M1", "R4"]
arms = {"F": "pointwise", "T": "Galerkin"}
arm_colours = {"F": "C0", "T": "C3"}
nbasis_all = [2, 4, 8, 16, 64]          # NXGRID 32; NBASIS 64 at NXGRID 65 is the converged self-reference
nbasis_low = [2, 4, 8, 16]
overshoot_tolerance = 0.10              # gamma above the reference by more than 10% is an overshoot
extent_change = np.log10(2)             # a dominant-mode extent change by more than x2 counts as a change of identity
fraction = 0.95                         # pyro Extent: theta range holding 95% of |phi|^2
ky_edges = np.array([0.0, 0.1, 0.3, 0.6, 2.0])
ky_labels = ["<0.1", "0.1-0.3", "0.3-0.6", ">0.6"]
fs_title, fs_legend = 16, 12


def log_xticks(ax, ticks):
    lims = ax.get_xlim()
    ax.set_xticks(ticks)
    ax.set_xlim(lims)
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
    ax.minorticks_off()

## GFTM leaves

One `PyroHypercube` per leaf; pyro's `Extent` gives the 95% extent of $|\phi|^2$ per mode. The settings each leaf ran with
are GyroRun's `code_flags` in its `manifest.json` (audited against `out.gftm.localdump` in Fusion_PhD, README "Settings audit").
**Parity** of the dominant $\phi$: even fraction $E = \sum|\phi(\theta)+\phi(-\theta)|^2 / (\sum|\phi(\theta)+\phi(-\theta)|^2 +
\sum|\phi(\theta)-\phi(-\theta)|^2)$ on GFTM's symmetric output grid, without pyro's $\sqrt{g_{\theta\theta}}$ weight
(`FieldLine.compute_linear_parity` needs a `Pyro` per sample). Every Hermite function has exact parity, so a GFTM mode has
$E = 0$ or 1 whatever the weight, up to that interpolation (printed below).

In [ ]:
warnings.filterwarnings("ignore")
gftm = {}       # (case, arm, NBASIS, WIDTH) -> dominant-mode arrays over the population
names = {}
for case in cases:
    for leaf in sorted((gftm_root / case / "kbm_free").glob("gal*_nb*")):
        gftm_scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
        gftm_scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
        Extent(gftm_scan.gk_output, fraction=fraction)
        flags = json.loads((leaf / "manifest.json").read_text())["code_flags"]
        key = (case, "T" if flags["FIELD_GALERKIN"] else "F", int(flags["NBASIS_MAX"]), round(float(flags["WIDTH"]), 4))
        assert leaf.name.startswith(f"gal{key[1]}_") and f"_nb{key[2]}_" in leaf.name and float(flags["FILTER"]) == 0.5
        gftm_output = gftm_scan.gk_output.data.set_index(sample="sample_name")
        names.setdefault(case, gftm_output.sample.values.astype(str))
        gftm_output = gftm_output.sel(sample=names[case])
        growth = gftm_output.growth_rate.pint.dequantify()
        # IBRANCH -1: GFTM ranks the modes by growth rate, so mode 0 is the dominant one
        assert int(flags["IBRANCH"]) == -1 and (growth.fillna(-np.inf).argmax("mode") == 0).all()
        dominant = gftm_output.isel(mode=0)
        gamma = growth.isel(mode=0).clip(min=0).fillna(0).values           # no unstable mode -> 0
        unstable = gamma > 0
        theta = np.asarray(gftm_output.theta.values, dtype=float)
        phi = dominant.eigenfunctions.sel(field="phi").pint.dequantify().transpose("sample", "theta").values
        # the grid is symmetric only to ~0.01 rad, so phi(-theta) is interpolated (as pyro's compute_linear_parity does)
        phi_flipped = np.array([np.interp(-theta, theta, row.real) + 1j * np.interp(-theta, theta, row.imag) for row in phi])
        even = (np.abs(phi + phi_flipped) ** 2).sum(1)
        even_fraction = even / (even + (np.abs(phi - phi_flipped) ** 2).sum(1))
        gftm[key] = dict(gamma=gamma,
                         omega=np.where(unstable, dominant.mode_frequency.pint.dequantify().values, np.nan),
                         extent=np.where(unstable, dominant.extent.sel(field="phi").pint.dequantify().values, np.nan),
                         even=np.where(unstable, even_fraction, np.nan),
                         units=str(gftm_output.growth_rate.data.units))
        if key[2] == 64:
            gftm[key]["phi"], gftm[key]["theta"] = phi, theta
widths = np.array(sorted({key[3] for key in gftm}))
print("leaves:", pd.Series([key[0] for key in gftm]).value_counts().to_dict(), "| widths:", len(widths), widths.min(), "-", widths.max())
print("population:", {case: len(names[case]) for case in cases})
even_all = np.concatenate([value["even"] for value in gftm.values()])
even_all = even_all[np.isfinite(even_all)]
print("dominant-mode even fraction: share within 0.01 of 0 or 1:", np.mean(np.minimum(even_all, 1 - even_all) < 0.01).round(4))


def stack(case, arm, nbasis, quantity):
    # (width, sample) array of one quantity at one (case, arm, NBASIS)
    return np.array([gftm[case, arm, nbasis, width][quantity] for width in widths])

## GS2 reference

The `kbm_free` cube of each case, samples paired with GFTM by `sample_name`. The cube holds the time-averaged $|\phi|^2$
(`eigenfunctions_squared`) on the union of the samples' $\theta$ grids (NaN outside a sample's own grid), not complex
$\phi$; the extent is pyro's `Extent` rule applied to each sample's own grid.

In [ ]:
gs2 = {}
for case in cases:
    gs2_scan = PyroHypercube(pyroscan_json=gs2_root / case / "kbm_free" / "pyroscan.json", load_base_pyro=True)
    gs2_scan.load_gk_output(netcdf_file=gs2_root / case / "kbm_free" / "pyroscan.nc")
    gs2_output = gs2_scan.gk_output.data.set_index(sample="sample_name").sel(sample=names[case])
    # same convention (vref_nrl / lref_minor_radius); a cube may tag its units with its normalisation's name
    units = {re.sub(r"_pyroscan_base\d+", "", str(gs2_output.growth_rate.data.units))} | \
            {re.sub(r"_pyroscan_base\d+", "", gftm[key]["units"]) for key in gftm if key[0] == case}
    assert len(units) == 1, units
    phi_squared = gs2_output.eigenfunctions_squared.sel(field="phi").isel(kx=0).pint.dequantify()
    phi_squared = [phi_squared.isel(sample=i).dropna("theta") for i in range(len(names[case]))]
    gamma_gs2 = gs2_output.growth_rate.pint.dequantify().values
    assert (gamma_gs2 > 0).all() and (gs2_output.growth_rate_tolerance.pint.dequantify().values < 0.05).all()
    gs2[case] = dict(gamma=gamma_gs2, omega=gs2_output.mode_frequency.pint.dequantify().values,
                     ky=gs2_output.ky.pint.dequantify().values,
                     phi_squared=phi_squared,
                     extent=np.array([float(Extent._compute(power, fraction)[0]) for power in phi_squared]))
    print(case, "| GS2-KBM samples:", len(names[case]), "| median phi extent:", np.median(gs2[case]["extent"]).round(2),
          "| theta box:", round(float(max(power.theta.max() for power in phi_squared)), 1),
          "| ky bins:", np.bincount(np.digitize(gs2[case]["ky"], ky_edges) - 1, minlength=4).tolist())

## $W_{\rm shape}$: the width whose $N$-function basis best represents the GS2 $\phi(\theta)$

GFTM's basis functions in $\theta$ are $h_j(\theta/W)/\sqrt{W}$, $j < N$, with $h_j$ the orthonormal Hermite functions
($h_0 \propto e^{-x^2/2}$; GFTM's nodes are $\theta_i = W x_i$). For each width the captured fraction is
$C_N(W) = \sum_{j<N} |\langle h_j(\cdot/W)/\sqrt{W}, \phi\rangle|^2 / \int|\phi|^2 d\theta$, the share of $\int|\phi|^2$ held
by the best $N$-function approximation; $W_{\rm shape}(N) = \arg\max_W C_N(W)$ on the 24-width grid. The inner products
are integrals on a uniform 20001-point grid over the sample's own $\theta$ range (30+ points per oscillation of $h_{63}$
at $W = 0.2$), not GFTM's quadrature.

**Approximation:** the GS2 cube holds $|\phi|^2$ only, so the projection is of $|\phi|$, which drops $\phi$'s phase along
$\theta$. The check below repeats the selection on GFTM's own NBASIS-64 $\phi$, once complex and once $|\phi|$.

In [ ]:
def hermite_functions(n_basis, x):
    # orthonormal Hermite functions h_0 .. h_{n_basis-1} at x, by the stable three-term recursion
    h = np.zeros((n_basis, len(x)))
    h[0] = np.pi ** -0.25 * np.exp(-x ** 2 / 2)
    h[1] = np.sqrt(2) * x * h[0]
    for j in range(2, n_basis):
        h[j] = np.sqrt(2 / j) * x * h[j - 1] - np.sqrt((j - 1) / j) * h[j - 2]
    return h


def captured_fraction(theta, field):
    # C_N(W) for N = 1..64, every grid width: (width, 64) array
    theta_uniform = np.linspace(theta.min(), theta.max(), 20001)
    field_uniform = np.interp(theta_uniform, theta, field.real) + 1j * np.interp(theta_uniform, theta, field.imag)
    norm = np.trapezoid(np.abs(field_uniform) ** 2, theta_uniform)
    captured = []
    for width in widths:
        coefficients = np.trapezoid(hermite_functions(64, theta_uniform / width) * field_uniform, theta_uniform, axis=1) / np.sqrt(width)
        captured.append(np.cumsum(np.abs(coefficients) ** 2) / norm)
    return np.array(captured)


def w_shape(theta, field):
    captured = captured_fraction(theta, field)
    return {nbasis: widths[captured[:, nbasis - 1].argmax()] for nbasis in nbasis_all}


for case in cases:
    shapes = [w_shape(power.theta.values, np.sqrt(power.values) + 0j) for power in gs2[case]["phi_squared"]]
    gs2[case]["w_shape"] = {nbasis: np.array([shape[nbasis] for shape in shapes]) for nbasis in nbasis_all}
    print(case, "| median W_shape:", {nbasis: float(np.median(gs2[case]["w_shape"][nbasis])) for nbasis in nbasis_all})

# check: drop the phase of GFTM's converged phi (Galerkin, NBASIS 64, at the width closest to 1)
width_check = widths[np.abs(widths - 1).argmin()]
phase_check = []
for case in cases:
    entry = gftm[case, "T", 64, width_check]
    for phi, unstable in zip(entry["phi"], entry["gamma"] > 0):
        if unstable:
            complex_shape, magnitude_shape = w_shape(entry["theta"], phi), w_shape(entry["theta"], np.abs(phi) + 0j)
            phase_check += [dict(case=case, nbasis=nbasis, complex=complex_shape[nbasis], magnitude=magnitude_shape[nbasis])
                            for nbasis in nbasis_low]
phase_check = pd.DataFrame(phase_check)
phase_check["dex"] = np.abs(np.log10(phase_check.magnitude / phase_check.complex))
grid_step = np.median(np.diff(np.log10(widths)))
print(f"W_shape from |phi| vs complex phi (GFTM NB64, W {width_check:g}): median |log10 ratio| by NBASIS, and share within one grid step ({grid_step:.3f} dex)")
print(phase_check.groupby(["case", "nbasis"]).dex.agg(median="median", within_step=lambda d: np.mean(d <= grid_step * 1.01)).round(3).to_string())
fig_phase, axes_phase = plt.subplots(1, len(nbasis_low), figsize=(22, 5.5), sharex=True, sharey=True)
for ax, nbasis in zip(axes_phase, nbasis_low):
    for case, marker in zip(cases, "osd"):
        d = phase_check[(phase_check.case == case) & (phase_check.nbasis == nbasis)]
        jitter = 10 ** np.random.default_rng(0).uniform(-0.015, 0.015, len(d))
        ax.plot(d.complex * jitter, d.magnitude, ls="", marker=marker, alpha=0.4, label=case)
    ax.plot(widths[[0, -1]], widths[[0, -1]], color="k", ls="--", marker="")
    ax.set(xscale="log", yscale="log", xlabel=r"$W_{\rm shape}$ from complex $\phi$")
    ax.set_title(f"NBASIS {nbasis}", fontsize=fs_title)
    log_xticks(ax, [0.2, 0.5, 1, 2, 4])
axes_phase[0].set_ylabel(r"$W_{\rm shape}$ from $|\phi|$")
axes_phase[0].legend(fontsize=fs_legend)
plt.show()

## 1. Overshoot rate

Share of (sample, width) pairs at which $\gamma(W, N)$ exceeds the reference by more than 10%. References: (i) the arm's
own converged maximum $\max_W \gamma(W, 64)$, (ii) $\gamma_{\rm GS2}$. Under (i) the max-$\gamma$ selection can only pick a
spurious width if some width overshoots, so (i) at NBASIS 64 is 0 by construction.

In [ ]:
rows = []
for case in cases:
    for arm in arms:
        converged_max = stack(case, arm, 64, "gamma").max(0)
        for nbasis in nbasis_all:
            gamma_nb = stack(case, arm, nbasis, "gamma")
            rows.append(dict(case=case, arm=arm, nbasis=nbasis,
                             overshoot_own=np.mean(gamma_nb > (1 + overshoot_tolerance) * converged_max),
                             overshoot_gs2=np.mean(gamma_nb > (1 + overshoot_tolerance) * gs2[case]["gamma"]),
                             any_width_own=np.mean((gamma_nb > (1 + overshoot_tolerance) * converged_max).any(0))))
overshoot = pd.DataFrame(rows)
print(overshoot.round(3).to_string(index=False))

## 2. Width selection

$W_{\rm sel}(N) = \arg\max_W \gamma(W, N)$ (NaN if no width has an unstable mode). Compared with
$W_\gamma(N)$, the width whose $\gamma(W, N)$ is closest to GS2 (an oracle: it consults the answer), and $W_{\rm shape}(N)$
above; plus $W_{\rm sel}(64)$ of the same arm (does the low-NBASIS search land where the converged search does?). Each is a
median $|\log_{10}|$ ratio over the samples where $W_{\rm sel}$ exists. The grid step is 0.06-0.08 dex.

In [ ]:
selection, rows = {}, []
for case in cases:
    samples = np.arange(len(names[case]))
    for arm in arms:
        gamma_64 = stack(case, arm, 64, "gamma")
        for nbasis in nbasis_all:
            gamma_nb = stack(case, arm, nbasis, "gamma")
            pick = gamma_nb.argmax(0)
            found = gamma_nb.max(0) > 0
            selection[case, arm, nbasis] = dict(
                pick=pick, found=found, w_sel=np.where(found, widths[pick], np.nan),
                w_gamma=widths[np.abs(gamma_nb - gs2[case]["gamma"]).argmin(0)])
        for nbasis in nbasis_all:
            sel = selection[case, arm, nbasis]
            rows.append(dict(case=case, arm=arm, nbasis=nbasis, n_found=int(sel["found"].sum()),
                             to_w_gamma=np.nanmedian(np.abs(np.log10(sel["w_sel"] / sel["w_gamma"]))),
                             to_w_shape=np.nanmedian(np.abs(np.log10(sel["w_sel"] / gs2[case]["w_shape"][nbasis]))),
                             to_w_sel64=np.nanmedian(np.abs(np.log10(sel["w_sel"] / selection[case, arm, 64]["w_sel"]))),
                             signed_to_w_gamma=np.nanmedian(np.log10(sel["w_sel"] / sel["w_gamma"])),
                             # a selection on the first or last grid width is the search hitting the range, not a maximum
                             sel_at_edge=np.mean(np.isin(sel["w_sel"], widths[[0, -1]])),
                             w_gamma_at_edge=np.mean(np.isin(sel["w_gamma"], widths[[0, -1]])),
                             same_as_w_gamma=np.mean(sel["w_sel"] == sel["w_gamma"])))
width_selection = pd.DataFrame(rows)
print(width_selection.round(3).to_string(index=False))

## 3. Accuracy at the selected width

Against GS2, over the population (a sample with no unstable mode counts as $\gamma = 0$):
- `logmed_gamma`: $\log_{10}$ median $|\gamma(W_{\rm sel}(N), N) - \gamma_{\rm GS2}|$;
- `two_stage_16`, `two_stage_64`: the same for the re-solve at NBASIS 16 (NXGRID 32) or 64 (NXGRID 65) at $W_{\rm sel}(N)$;
- `oracle`: at $W_\gamma(N)$, a bound, never an accuracy;
- `extent_ratio`: median $|\log_{10}|$ (GFTM $\phi$ extent at $W_{\rm sel}(N)$, NBASIS $N$) / (GS2 $\phi$ extent)| (pyro `Extent`, 95%).

In [ ]:
rows = []
for case in cases:
    samples = np.arange(len(names[case]))
    gamma_gs2 = gs2[case]["gamma"]
    logmed = lambda gamma: np.log10(np.median(np.abs(gamma - gamma_gs2)))
    for arm in arms:
        for nbasis in nbasis_all:
            sel = selection[case, arm, nbasis]
            at_sel = lambda nb, quantity="gamma": stack(case, arm, nb, quantity)[sel["pick"], samples]
            at_gamma = stack(case, arm, nbasis, "gamma")[np.searchsorted(widths, sel["w_gamma"]), samples]
            rows.append(dict(case=case, arm=arm, nbasis=nbasis, logmed_gamma=logmed(at_sel(nbasis)),
                             two_stage_16=logmed(at_sel(16)), two_stage_64=logmed(at_sel(64)), oracle=logmed(at_gamma),
                             extent_ratio=np.nanmedian(np.abs(np.log10(at_sel(nbasis, "extent") / gs2[case]["extent"])))))
accuracy = pd.DataFrame(rows)
print(accuracy.round(3).to_string(index=False))

## 4. Mode identity

The dominant mode's signature is (sign of $\omega$, parity, extent). It **changes** between two settings when the sign of
$\omega$ or the parity differs, or the extent differs by more than $\times 2$; pairs where either setting has no unstable mode
are left out. Two comparisons per (case, arm, $N$): **with $N$**, $(W, N)$ against $(W, 64)$ at every width; **with $W$**,
$(W_i, N)$ against $(W_{i+1}, N)$, neighbouring grid widths. Binned in GS2 ky (the laptop failure was at ky 0.3/0.6).

In [ ]:
def changed(a, b):
    # (component shares, any-change mask) between two (width, sample) signature sets, NaN where either has no mode
    both = np.isfinite(a["omega"]) & np.isfinite(b["omega"])
    sign = np.sign(a["omega"]) != np.sign(b["omega"])
    parity = (a["even"] > 0.5) != (b["even"] > 0.5)
    extent = np.abs(np.log10(a["extent"] / b["extent"])) > extent_change
    return both, sign & both, parity & both, extent & both, (sign | parity | extent) & both


rows = []
for case in cases:
    ky_bin = np.digitize(gs2[case]["ky"], ky_edges) - 1
    for arm in arms:
        signature = {nbasis: {q: stack(case, arm, nbasis, q) for q in ("omega", "even", "extent")} for nbasis in nbasis_all}
        for nbasis in nbasis_all:
            comparisons = {"with_N": (signature[nbasis], signature[64]),
                           "with_W": ({q: v[:-1] for q, v in signature[nbasis].items()}, {q: v[1:] for q, v in signature[nbasis].items()})}
            for comparison, (a, b) in comparisons.items():
                both, sign, parity, extent, any_change = changed(a, b)
                for k, label in enumerate(["all"] + ky_labels):
                    in_bin = np.ones(len(ky_bin), bool) if label == "all" else ky_bin == k - 1
                    n_pairs = both[:, in_bin].sum()
                    rows.append(dict(case=case, arm=arm, nbasis=nbasis, comparison=comparison, ky=label, n_pairs=int(n_pairs),
                                     any=any_change[:, in_bin].sum() / max(n_pairs, 1), sign=sign[:, in_bin].sum() / max(n_pairs, 1),
                                     parity=parity[:, in_bin].sum() / max(n_pairs, 1), extent=extent[:, in_bin].sum() / max(n_pairs, 1)))
identity = pd.DataFrame(rows)
print(identity[identity.ky == "all"].round(3).to_string(index=False))
print("\nany-change share by ky bin:")
print(identity[identity.ky != "all"].pivot_table(index=["comparison", "case", "nbasis"], columns=["ky", "arm"], values="any")[
    [(ky, arm) for ky in ky_labels for arm in arms]].round(3).to_string())

## Figures

fig1-4: one panel per case, x = NBASIS (log), blue = pointwise (`FIELD_GALERKIN F`), red = Galerkin (`T`); both arms on
the same samples.
- fig1: overshoot rate; solid = against the arm's own converged maximum, dashed = against GS2.
- fig2: median $|\log_{10}|$ width ratio; solid = $W_{\rm sel}/W_\gamma$, dashed = $W_{\rm sel}/W_{\rm shape}$,
  dotted = $W_{\rm sel}(N)/W_{\rm sel}(64)$.
- fig3: LogMed $|\gamma - \gamma_{\rm GS2}|$ at $W_{\rm sel}(N)$; solid = at NBASIS $N$, dashed = two-stage re-solve at
  NBASIS 16, dotted = oracle $W_\gamma$ (a bound). fig3b: median $|\log_{10}|$ extent ratio at $W_{\rm sel}(N)$.
- fig4: share of dominant-mode identity changes vs GS2 ky bin; rows = case, columns = NBASIS; solid = with $N$ (against
  NBASIS 64 at the same width), dashed = with $W$ (neighbouring widths).
- fig5 (format copied from the GFTM fig5 of `GFTM_numerics/gftm_gauss_hermite_extent_and_omega_vs_width.ipynb`, `per_nbasis`):
  LogMed $|\gamma - \gamma_{\rm GS2}|$ vs WIDTH; rows = case, columns = NBASIS.

In [ ]:
def per_case(table, styles, ylabel):
    fig, axes = plt.subplots(1, len(cases), figsize=(19, 5.5), sharex=True, sharey=True, squeeze=False)
    for ax, case in zip(axes[0], cases):
        for arm in arms:
            d = table[(table.case == case) & (table.arm == arm)].sort_values("nbasis")
            for column, (ls, label) in styles.items():
                ax.plot(d.nbasis, d[column], color=arm_colours[arm], ls=ls, marker="o",
                        label=f"{arms[arm]}, {label}" if case == cases[0] else None)
        ax.set(xscale="log", xlabel="NBASIS")
        ax.set_title(case, fontsize=fs_title)
        log_xticks(ax, nbasis_all)
    axes[0, 0].set_ylabel(ylabel)
    axes[0, 0].legend(fontsize=fs_legend)
    plt.show()
    return fig


fig1 = per_case(overshoot, {"overshoot_own": ("-", "vs own NB64 max"), "overshoot_gs2": ("--", "vs GS2")},
                r"overshoot share ($\gamma > 1.1\,\gamma_{\rm ref}$)")
fig2 = per_case(width_selection, {"to_w_gamma": ("-", r"$W_\gamma$"), "to_w_shape": ("--", r"$W_{\rm shape}$"),
                                  "to_w_sel64": (":", r"$W_{\rm sel}(64)$")}, r"Med $|\log_{10} W_{\rm sel}/W_{\rm ref}|$")
fig3 = per_case(accuracy, {"logmed_gamma": ("-", "at NBASIS N"), "two_stage_16": ("--", "re-solve NB16"), "oracle": (":", r"oracle $W_\gamma$")},
                r"LogMed $|\gamma-\gamma_{\rm GS2}|$ at $W_{\rm sel}$")
fig3b = per_case(accuracy, {"extent_ratio": ("-", r"$\phi$ extent")}, r"Med $|\log_{10}$ Extent Ratio$|$ at $W_{\rm sel}$")

fig4, axes4 = plt.subplots(len(cases), len(nbasis_low), figsize=(22, 13), sharex=True, sharey=True, squeeze=False)
for r, case in enumerate(cases):
    for j, nbasis in enumerate(nbasis_low):
        ax = axes4[r, j]
        for arm in arms:
            for comparison, ls in (("with_N", "-"), ("with_W", "--")):
                d = identity[(identity.case == case) & (identity.arm == arm) & (identity.nbasis == nbasis) &
                             (identity.comparison == comparison) & (identity.ky != "all")]
                ax.plot(range(len(ky_labels)), d["any"], color=arm_colours[arm], ls=ls, marker="o",
                        label=f"{arms[arm]}, {comparison.replace('_', ' ')}" if (r, j) == (0, 0) else None)
        ax.set_title(f"{case}, NBASIS {nbasis}", fontsize=fs_title)
        ax.set_xticks(range(len(ky_labels)), ky_labels)
        if r == len(cases) - 1:
            ax.set_xlabel(r"GS2 $k_y\rho$")
    axes4[r, 0].set_ylabel("identity-change share")
axes4[0, 0].legend(fontsize=fs_legend)
plt.show()

rows = []
for case in cases:
    for (case_, arm, nbasis, width), entry in gftm.items():
        if case_ == case:
            rows.append(dict(case=case, arm=arm, nb=nbasis, W=width,
                             gamma_logmed=np.log10(np.median(np.abs(entry["gamma"] - gs2[case]["gamma"])))))
acc = pd.DataFrame(rows).sort_values(["case", "arm", "nb", "W"])
fig5, axes5 = plt.subplots(len(cases), len(nbasis_all), figsize=(22, 13), sharex=True, sharey=True, squeeze=False)
for r, case in enumerate(cases):
    for j, nb in enumerate(nbasis_all):
        ax = axes5[r, j]
        for arm in arms:
            a = acc[(acc.case == case) & (acc.arm == arm) & (acc.nb == nb)]
            ax.plot(a.W, a.gamma_logmed, color=arm_colours[arm], label=arms[arm])
        ax.set(xscale="log")
        ax.set_title(f"{case}, NBASIS {nb}", fontsize=fs_title)
        if r == len(cases) - 1:
            ax.set_xlabel("WIDTH")
            log_xticks(ax, [0.2, 0.5, 1, 2, 4])
    axes5[r, 0].set_ylabel(r"LogMed $|\gamma_{\rm GFTM}-\gamma_{\rm GS2}|$")
axes5[0, 0].legend(fontsize=fs_legend)
plt.show()

In [ ]:
out = ROOT / "Plots" / analysis_name
out.mkdir(parents=True, exist_ok=True)
for name, fig in (("fig0_w_shape_phase_check", fig_phase), ("fig1_overshoot_vs_nbasis", fig1), ("fig2_width_selection_vs_nbasis", fig2),
                  ("fig3_gamma_err_at_selected_width", fig3), ("fig3b_extent_ratio_at_selected_width", fig3b),
                  ("fig4_mode_identity_vs_ky", fig4), ("fig5_gamma_err_vs_width_per_nbasis", fig5)):
    fig.savefig(out / f"{name}.png", bbox_inches="tight")
# medians and shares only, no per-sample rows
for name, table in (("overshoot", overshoot), ("width_selection", width_selection), ("accuracy_at_selected_width", accuracy),
                    ("mode_identity", identity), ("accuracy_vs_width", acc)):
    table.round(4).to_csv(out / f"{name}.csv", index=False)
print(sorted(p.name for p in out.iterdir()))